#### Q: Let's start with `Finance Flow`

Build a multi-agent financial assistant that:

- Analyzes spending (from uploaded CSVs or manual input)
- Identifies patterns and offers insights
- Generates monthly budgets
- Provides personalized financial advice
- Allows multi-step workflows and memory

This is ideal to test:

- agent coordination
- branching logic
- structured output
- tool calling
- persistence


## High Level Architecture

|          Agent          | Responsibility                                                   |
|:-----------------------:|:-----------------------------------------------------------------|
|      Input Router       | Detect user intent → route to the correct workflow               |
| Spending Analyzer Agent | Parse transactions, categorize spending, compute totals          |
|      Insight Agent      | Identify habits/trends (overspending categories, risky patterns) |
|      Advice Agent       | Generate actionable suggestions (budgeting, savings, goals)      |
| Report Generator Agent  | Format outputs into reports (weekly/monthly)                     |
|       Memory Node       | Store past spending summaries or user preferences                |

## Prompt for Agents

Input Router

```
You are an intent classifier for a finance system.
Classify the user's message into one of:
["analyze_spending", "financial_advice", "upload_data", "budget_planning", "other"]
Return JSON:
{"intent": "..."}
```
Spending Analyzer
```
You are a financial data processor.
Input: raw text or CSV rows.
Output: categorized transactions in JSON. If unsure, guess category by description.
```

Insight Agent

```
Analyze spending patterns: overspending areas, category ratios, and trends.
Output JSON with 3 sections:
1. Key insights
2. Potential risks
3. Notable patterns
```

Advice Agent

```
Based on insights + user profile, generate practical personal finance suggestions.
Focus on saving, debt control, subscription trimming, and spending alignment with goals.
Format as: {"advice": [...]}
```

Report Generator

```
Convert insights + advice into a friendly financial report in Markdown.
Include:
- Summary
- Category pie insights
- Action plan
- One motivational tip
```

## Data Modelling

```json
{
  "date": "2026-04-01",
  "description": "Starbucks",
  "amount": 4.95,
  "category": "Food & Beverage",
  "type": "expense"
}
```


```json
{
  "income": 3200,
  "expenses": 2280,
  "savings_rate": 28.75,
  "largest_categories": [
    {"category": "Dining", "amount": 320},
    {"category": "Subscriptions", "amount": 145}
  ]
}

```

# Setup local environment

In [1]:

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.tools import tool
from langchain_core.tracers import ConsoleCallbackHandler
from langchain_openai import ChatOpenAI


llm = ChatOpenAI(
    base_url="http://localhost:8080/v1",
    model="unsloth/gemma-4-E4B-it-GGUF:Q4_K_M",
    temperature=0.5,
    api_key="not-needed-for-local-inference",
    callbacks=[ConsoleCallbackHandler()] # for debugging purpose
)

In [26]:
from typing import TypedDict, List, Any, Literal
from langchain_core.messages import HumanMessage, SystemMessage
from langchain.agents import create_agent
from langchain_core.tools import tool
from datetime import date

# ---- Tools ---
@tool("open_file", description="Open a file")
def open_file(file_path: str) -> str:
    with open(file_path, "r") as f:
        return f.read()

# ---- Functions -----
def classify_intent(user_input: str) -> str:
    classify_agent = create_agent(
        model=llm,
        system_prompt=SystemMessage(content="""
You are an intent classifier for a finance system.
Classify the user's message into one of:
["analyze_spending", "financial_advice", "upload_data", "budget_planning", "other"]
Return String represent the intent
        """)
    )
    return classify_agent.invoke({"messages": [
        HumanMessage(content=user_input)
    ]})["messages"][-1].content # return the content of the last message


class Transaction(TypedDict):
    date: date
    description: str
    amount: float
    category: str
    type: Literal["income", "expense"]

class TransactionList(TypedDict):
    transactions: List[Transaction]

def parse_transactions(user_input: str) -> TransactionList:
    parse_agent = create_agent(
        model = llm,
        tools = [open_file],
        response_format=TransactionList,
        system_prompt = SystemMessage(content="""
You are a financial data processor.
Input: raw text or CSV rows.
Output: categorized transactions in JSON. If unsure, guess category by description.
        """)
    )
    resp: TransactionList = parse_agent.invoke({"messages": [
        HumanMessage(content=user_input)
    ]})["structured_response"]
    return resp


def generate_insights(transactions: TransactionList) -> dict:
    return llm.invoke([
        SystemMessage(content="""
Analyze spending patterns: overspending areas, category ratios, and trends.
Output JSON with 3 sections:
1. Key insights
2. Potential risks
3. Notable patterns
        """),
        HumanMessage(content=f"Transactions: {transactions}")
    ]).content

def generate_advice(insights: dict) -> dict:
    return llm.invoke([
        SystemMessage(content="""
Based on insights + user profile, generate practical personal finance suggestions.
Focus on saving, debt control, subscription trimming, and spending alignment with goals.
Format as: {"advice": [...]}
        """),
        HumanMessage(content=f"Insights: {insights}")
    ]).content

def format_report(insights, advice) -> str:
    return llm.invoke([
        SystemMessage(content="""
Convert insights + advice into a friendly financial report in Markdown.
Include:
- Summary
- Category pie insights
- Action plan
- One motivational tip
        """),
        HumanMessage(content=f"Insights: {insights}\nAdvice: {advice}")
    ]).content

In [ ]:
router_response = classify_intent("I want to analyze my spending")

In [27]:
parsing_response = parse_transactions("""
Here is my monthly spending data

```csv
Date,Description,Amount,Category,Type
2023-01-01,Groceries,12.99,Food & Beverage,Expense
2023-01-15,Rent,1000,Housing,Income
2023-02-01,Groceries,15.99,Food & Beverage,Expense
```

""")

[llm/start] [llm:ChatOpenAI] Entering LLM run with input:
{
  "prompts": [
    "System: \nYou are a financial data processor.\nInput: raw text or CSV rows.\nOutput: categorized transactions in JSON. If unsure, guess category by description.\n        \nHuman: \nHere is my monthly spending data\n\n```csv\nDate,Description,Amount,Category,Type\n2023-01-01,Groceries,12.99,Food & Beverage,Expense\n2023-01-15,Rent,1000,Housing,Income\n2023-02-01,Groceries,15.99,Food & Beverage,Expense\n```"
  ]
}
[llm/end] [llm:ChatOpenAI] [42.51s] Exiting LLM run with output:
{
  "generations": [
    [
      {
        "text": "",
        "generation_info": {
          "finish_reason": "tool_calls",
          "logprobs": null
        },
        "type": "ChatGeneration",
        "message": {
          "lc": 1,
          "type": "constructor",
          "id": [
            "langchain",
            "schema",
            "messages",
            "AIMessage"
          ],
          "kwargs": {
            "content"

In [3]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, List


# ---- State -----
class FinanceState(TypedDict):
    user_input: str
    intent: str
    transactions: List[dict]
    insights: dict
    advice: dict
    report: str

# ---- Nodes -----
def router_node(state: FinanceState):
    # Call LLM to classify intent
    intent = classify_intent(state["user_input"])
    state["intent"] = intent
    return state

def analyze_node(state: FinanceState):
    tx = parse_transactions(state["user_input"])
    state["transactions"] = tx
    return state

def insight_node(state: FinanceState):
    ins = generate_insights(state["transactions"])
    state["insights"] = ins
    return state

def advice_node(state: FinanceState):
    adv = generate_advice(state["insights"])
    state["advice"] = adv
    return state

def report_node(state: FinanceState):
    rpt = format_report(state["insights"], state["advice"])
    state["report"] = rpt
    return state

# ---- Graph -----
graph = StateGraph(FinanceState)

graph.add_node("router", router_node)
graph.add_node("analyze", analyze_node)
graph.add_node("insight", insight_node)
graph.add_node("advice", advice_node)
graph.add_node("report", report_node)

graph.set_entry_point("router")

graph.add_conditional_edges(
    "router",
    lambda s: s["intent"],
    {
        "analyze_spending": "analyze",
        "financial_advice": "advice",
        "budget_planning": "advice",
        "upload_data": "analyze",
        "other": END
    }
)

graph.add_edge("analyze", "insight")
graph.add_edge("insight", "advice")
graph.add_edge("advice", "report")
graph.add_edge("report", END)

flow = graph.compile()

In [28]:

from langchain_core.tools import tool
import pandas as pd

@tool
def load_csv_transactions(file_path: str) -> dict:
    """Read a CSV file of transactions and return summary stats."""
    df = pd.read_csv(file_path)

    df["date"] = pd.to_datetime(df["date"], errors="coerce")

    return {
        "row_count": len(df),
        "total_amount": float(df["amount"].sum()),
        "categories": df["category"].unique().tolist(),
        "sample_rows": df.head().to_dict(orient="records")
    }


In [29]:

def spending_analyzer_node(state):
    file_obj = state.get("file")  # from user
    message = state["user_input"]

    if file_obj:
        # convert to agent input
        return {
            "messages": [
                {
                    "role": "user",
                    "content": f"Please analyze this file: {file_obj.name}"
                }
            ],
            "file_path": file_obj.name
        }


In [30]:
agent = create_agent(model=llm, tools=[load_csv_transactions])

In [31]:
file_path = "./tests/mock_transactions.csv"
result = agent.invoke({
    "messages": [
        {"role": "user", "content": f"Analyze this file: {file_path}"}
    ],
    "file_path": file_path
})


[llm/start] [llm:ChatOpenAI] Entering LLM run with input:
{
  "prompts": [
    "Human: Analyze this file: /home/trung_nguyendinh/Documents/llm-sideprojects/tests/mock_transactions.csv"
  ]
}
[llm/end] [llm:ChatOpenAI] [21.93s] Exiting LLM run with output:
{
  "generations": [
    [
      {
        "text": "",
        "generation_info": {
          "finish_reason": "tool_calls",
          "logprobs": null
        },
        "type": "ChatGeneration",
        "message": {
          "lc": 1,
          "type": "constructor",
          "id": [
            "langchain",
            "schema",
            "messages",
            "AIMessage"
          ],
          "kwargs": {
            "content": "",
            "additional_kwargs": {
              "refusal": null
            },
            "response_metadata": {
              "token_usage": {
                "completion_tokens": 278,
                "prompt_tokens": 99,
                "total_tokens": 377,
                "completion_tokens_de